# SQL vs. pandas Mini-tutorial

This tutorial is based on this YouTube Video: [SQL for Beginners in 15 minutes](https://www.youtube.com/watch?v=_vxobA36UN4).  
Please speed watch the video first, to get an overall sense of its content, then as you do the examples below, rewatch the specific portions, since you are replicating the examples in Python.

I have created the DB file, named "school_database.db", containing the two tables of the video. We also have access to the two tables as CSV files. Below we will show how to operate with data both in pandas and SQL.

**NOTE:** Throughout the tutorials there are **tasks** for you to complete, typically labeled as **Task for you**.

**Table of Content**

1. [Loading the data](#sec1)
2. [The `SELECT * FROM` command](#sec2)
3. [The `SELECT ... FROM` command](#sec3)
4. [The `SELECT ... FROM ... WHERE` command](#sec4)
5. [Sorting with `ORDER BY`](#sec5)
6. [Exercise: Combine all commands](#sec6)
7. [The GROUP BY command](#sec7)
8. [Filtering aggregated data with `HAVING`](#sec8)
9. [Utility keywords](#sec9)
10. [Combining Tables]()

<a id="sec1"></a>
## 1. Loading the data

Load the two CSV tables that we will use with `pandas`

In [1]:
import pandas as pd
import numpy as np

### Load the table `students`

In [2]:
students_df = pd.read_csv('students.csv', 
                          index_col='id')
students_df.head()

,student_name,grade_level,gpa,school_lunch,birthday,email
id,,,,,,
1,Abby Johnson,10,3.1,Yes,5/14/08,abby.johnson@mavenhighschool.com
2,Bob Smith,11,3.1,No,9/30/07,bob.smith@mavenhighschool.com
3,Catherine Davis,12,3.6,Yes,11/21/06,catherine.davis@mavenhighschool.com
4,Daniel Brown,9,3.5,Yes,3/10/09,daniel.brown@mavenhighschool.com
5,Eva Martinez,10,2.7,No,2/5/08,eva.martinez@mavenhighschool.com


**Task for you:** Load the data without `index_col` parameter, what do you notice?

### Load the table `student_grades`

In [3]:
student_grades_df = pd.read_csv("student_grades.csv")
student_grades_df.head()

,semester_id,class_id,department,class_name,student_id,final_grade
0,SPR_2024,101,Math,Algebra,4,85.0
1,SPR_2024,101,Math,Algebra,8,76.0
2,SPR_2024,101,Math,Algebra,11,90.0
3,SPR_2024,101,Math,Algebra,15,97.0
4,SPR_2024,102,Math,Geometry,1,93.0


### Connect to the database

Python has its own built-in library that manages SQLite databases (one of the light-weight RDBMSs). This library is `sqlite3` and will execute the SQL queries for us.

In [4]:
import sqlite3

db_file = "school_database.db"

conn = sqlite3.connect(db_file) # you are connected to the database
cursor = conn.cursor() # you have appointed a "helper" that will perform SQL commands for you
cursor

<a id="sec2"></a>
## 2. The `SELECT * FROM` command

The command `SELECT * from students` gives us the whole table. 

In [5]:
query = "SELECt * from students"
cursor.execute(query) # the cursor goes to the location in the database where the data is
cursor.fetchall() # we have to explicetely ask for the data to be feteched

[(1,
  'Abby Johnson',
  10,
  3.1,
  'Yes',
  '2008-05-14',
  'abby.johnson@mavenhighschool.com'),
 (2,
  'Bob Smith',
  11,
  3.1,
  'No',
  '2007-09-30',
  'bob.smith@mavenhighschool.com'),
 (3,
  'Catherine Davis',
  12,
  3.6,
  'Yes',
  '2006-11-21',
  'catherine.davis@mavenhighschool.com'),
 (4,
  'Daniel Brown',
  9,
  3.5,
  'Yes',
  '2009-03-10',
  'daniel.brown@mavenhighschool.com'),
 (5,
  'Eva Martinez',
  10,
  2.7,
  'No',
  '2008-02-05',
  'eva.martinez@mavenhighschool.com'),
 (6,
  'Frank Wilson',
  11,
  3.2,
  'No',
  '2007-07-17',
  'frank.wilson@mavenhighschool.com'),
 (7,
  'Grace Lee',
  12,
  3.0,
  'Yes',
  '2006-12-02',
  'grace.lee@mavenhighschool.com'),
 (8,
  'Henry Taylor',
  9,
  3.0,
  'Yes',
  '2009-06-08',
  'henry.taylor@mavenhighschool.com'),
 (9,
  'Isabella Moore',
  10,
  2.8,
  'Yes',
  '2008-01-19',
  'isabella.moore@mavenhighschool.com'),
 (10,
  'Jack Thompson',
  11,
  2.9,
  'Yes',
  '2007-04-25',
  'jack.thompson@mavenhighschool.com'),
 (11

### Running SQL commands with pandas

Just like there are functions to read data from CSV, JSON, EXCEL, there is a function that takes a query and a connection to a database and reads the content:

In [6]:
results_df = pd.read_sql_query(query, conn)
results_df

,id,student_name,grade_level,gpa,school_lunch,birthday,email
0,1,Abby Johnson,10,3.1,Yes,2008-05-14,abby.johnson@mavenhighschool.com
1,2,Bob Smith,11,3.1,No,2007-09-30,bob.smith@mavenhighschool.com
2,3,Catherine Davis,12,3.6,Yes,2006-11-21,catherine.davis@mavenhighschool.com
3,4,Daniel Brown,9,3.5,Yes,2009-03-10,daniel.brown@mavenhighschool.com
4,5,Eva Martinez,10,2.7,No,2008-02-05,eva.martinez@mavenhighschool.com
5,6,Frank Wilson,11,3.2,No,2007-07-17,frank.wilson@mavenhighschool.com
6,7,Grace Lee,12,3.0,Yes,2006-12-02,grace.lee@mavenhighschool.com
7,8,Henry Taylor,9,3.0,Yes,2009-06-08,henry.taylor@mavenhighschool.com
8,9,Isabella Moore,10,2.8,Yes,2008-01-19,isabella.moore@mavenhighschool.com
9,10,Jack Thompson,11,2.9,Yes,2007-04-25,jack.thompson@mavenhighschool.com


<a id="sec3"></a>
## 3. The `SELECT ... FROM` command

Most of the time we don't want all the data, we only want a few colums. In that case we specify queries such as:

```
SELECT
    student_name,
    gpa
FROM
    students;
```

In pandas, this is equivalent to:

```students_df[['student_name', 'gpa']]```

In [7]:
students_df[['student_name', 'gpa']]

,student_name,gpa
id,,
1,Abby Johnson,3.1
2,Bob Smith,3.1
3,Catherine Davis,3.6
4,Daniel Brown,3.5
5,Eva Martinez,2.7
6,Frank Wilson,3.2
7,Grace Lee,3.0
8,Henry Taylor,3.0
9,Isabella Moore,2.8


Now let's do the same with sqlite3:

In [8]:
query = "SELECT student_name, gpa from students"
cursor.execute(query) 
cursor.fetchall()

[('Abby Johnson', 3.1),
 ('Bob Smith', 3.1),
 ('Catherine Davis', 3.6),
 ('Daniel Brown', 3.5),
 ('Eva Martinez', 2.7),
 ('Frank Wilson', 3.2),
 ('Grace Lee', 3.0),
 ('Henry Taylor', 3.0),
 ('Isabella Moore', 2.8),
 ('Jack Thompson', 2.9),
 ('Karen White', 3.4),
 ('Liam Green', 4.0),
 ('Mia Harris', 3.0),
 ('Noah Scott', None),
 ('Olivia Adams', 3.7),
 ('Peter Park', 2.9)]

And then again through pandas. **Note:** Because we already have a dataframe, this is redundant, but we are practicing for situation when you have access to a big database, and not the original CSVs. 

In [9]:
pd.read_sql_query(query, conn)

,student_name,gpa
0,Abby Johnson,3.1
1,Bob Smith,3.1
2,Catherine Davis,3.6
3,Daniel Brown,3.5
4,Eva Martinez,2.7
5,Frank Wilson,3.2
6,Grace Lee,3.0
7,Henry Taylor,3.0
8,Isabella Moore,2.8
9,Jack Thompson,2.9


**Task for you:** Write and execute a query that selects the columns student_name, birthday, and email.

<a id="sec4"></a>
## 4. The `SELECT ... FROM ... WHERE` command

This command allows us to do filtering of a table based on the values of the columns. Here is the SQL command:

```
SELECT
    *
FROM
    students
WHERE
    school_lunch = 'Yes';
```

In pandas this is equivlent to: 

```
students_df[students_df['school_lunch'] == 'Yes']
```

In [10]:
students_df[students_df['school_lunch'] == 'Yes']

,student_name,grade_level,gpa,school_lunch,birthday,email
id,,,,,,
1,Abby Johnson,10,3.1,Yes,5/14/08,abby.johnson@mavenhighschool.com
3,Catherine Davis,12,3.6,Yes,11/21/06,catherine.davis@mavenhighschool.com
4,Daniel Brown,9,3.5,Yes,3/10/09,daniel.brown@mavenhighschool.com
7,Grace Lee,12,3.0,Yes,12/2/06,grace.lee@mavenhighschool.com
8,Henry Taylor,9,3.0,Yes,6/8/09,henry.taylor@mavenhighschool.com
9,Isabella Moore,10,2.8,Yes,1/19/08,isabella.moore@mavenhighschool.com
10,Jack Thompson,11,2.9,Yes,4/25/07,jack.thompson@mavenhighschool.com
12,Liam Green,10,4.0,Yes,8/3/08,liam.green@mavenhighschool.com
15,Olivia Adams,9,3.7,Yes,12/11/09,olivia.adams@mavenhighschool.com


Similarly as above we can run this in sqlite3:

In [11]:
query = "SELECT * from students WHERE school_lunch = 'Yes'"
cursor.execute(query)
cursor.fetchone() # only show me the first line

(1,
 'Abby Johnson',
 10,
 3.1,
 'Yes',
 '2008-05-14',
 'abby.johnson@mavenhighschool.com')

**Task for you:** Filter the table students based on two criteria: school_lunch is "Yes" and the gpa is > than 3.3.

In [ ]:
# write the pandas code for filtering

In [ ]:
# write the SQL query and then execute, refer to video for syntax

## 5. Sorting with `ORDER BY`

The SQL query in this occasion is:

```
SELECT
    *
FROM
    students
ORDER BY
    gpa ASC;
```

This will sort the data in the ascending order:

In [ ]:
query = "SELECT * FROM students ORDER BY gpa ASC"
cursor.execute(query)
cursor.fetchall()

We can use DESC for the descending order:

In [ ]:
query = "SELECT * FROM students ORDER BY gpa DESC"
cursor.execute(query)
cursor.fetchall()

**Your Task:** Write pandas code to sort the dataframe first in ascending and then in decscending order. 

<a id="sec6"></a>
## 6. Combine all commands

Now that you saw the commands SELECT ... FROM, WHERE, and ORDER BY, write code both in SQL and in pandas to sort in descending orders students with no school lunch and gpa greater than 3.0. You should be able to find 3 such students. Fee free to write the Pandas one first, if it helps you get a sense of what you are doing. (The SQL solution is in the video.)

In [ ]:
# SQL solution

In [ ]:
# Pandas solution

<a id="sec7"></a>
## 7. The `GROUP BY` command

Suppose that now we want to calculate the average GPA by grade level. This is something that we learned to do with the `groupby` function in Pandas, but Pandas has borrowed that from SQL databases.

This is how we'll write this in SQL:

```
SELECT
    grade_level,
    AVG(gpa) AS average_gpa
FROM
    students
GROUP BY
    grade_level;
```

In [ ]:
query = "SELECT grade_level, avg(gpa) AS average_gpa FROM students GROUP BY grade_level"
cursor.execute(query)
cursor.fetchall()

**Task for you:** Solve the same problem but now using pandas's groupby function.

<a id="sec8"></a>
## 8. Filtering aggregated data with HAVING

Once we have grouped data by a certain criteria (for example the average gpa), we might want to further filter the data to focus only on a subset of it. For example, we might want to: **Filter groups where average GPA is below 3.3**

This is done by this SQL query:

```
SELECT
    grade_level,
    AVG(gpa) AS average_gpa
FROM
    students
GROUP BY
    grade_level
HAVING
    average_gpa < 3.3;
```

In [ ]:
query = "SELECT grade_level, AVG(gpa) AS average_gpa FROM students GROUP BY grade_level HAVING average_gpa < 3.3;"
cursor.execute(query)
cursor.fetchall()

**Task for you:** Repeat the same task but now with pandas. **Hint:** First create a new dataframe by using .reset_index with the name parameter to name the average_gpa as a column. Then, it will be easy to filter by the values of this column.

<a id="sec9"></a>
## 9. Utility Keywords

Here are three useful keywords in SQL for some useful tasks. Remember to watch the video for explanations on them.

* LIMIT
* COUNT
* DISTINCT

Your task will be to find the pandas equivalents for each of them once you have executed the SQL code.



In [ ]:
query = """
SELECT
    *
FROM
    students
LIMIT 5;"""
cursor.execute(query)
cursor.fetchall()

**Task for you:** Write the pandas equivalent. (It's very simple and we use it all the time.)

In [ ]:
query = """
SELECT
    COUNT(*)
FROM
    students;"""
cursor.execute(query)
cursor.fetchall()

**Task for you:** How can we get the number of rows for a dataframe in pandas?

In [ ]:
query = """
SELECT DISTINCT
    GPA
FROM
    students
ORDER BY
    GPA DESC;"""

cursor.execute(query)
cursor.fetchall()

**Task for you:** Find a way to get the same distinct values in pandas (there are different ways to do this).

<a id="sec10"></a>
## 10. Combining Tables

Combining tables is done through the so-called "left join". In pandas this is accomplished by using `.merge`. Let's see both cases. Again, first rewatch the video before running this snippet.

In [ ]:
query = """
SELECT
    s.student_name,
    sg.class_name,
    sg.final_grade
FROM
    students AS s
LEFT JOIN
    student_grades AS sg
ON
    s.id = sg.student_id;
"""
cursor.execute(query)
cursor.fetchall()

Pandas solution:

In [ ]:
merged_df = pd.merge(
    students_df,
    student_grades_df,
    how='left',           # Equivalent to LEFT JOIN
    left_on='id',         # Key from the left dataframe (students_df)
    right_on='student_id' # Key from the right dataframe (student_grades_df)
)

merged_df[['student_name', 'class_name', 'final_grade']]

<a id="sec11"></a>
## 11. Practice Problems

Try your best to solve these problems both with SQL and pandas.

**Problem 1:** Find the names and GPAs of all students in grade 10 who have a GPA above 3.0, sorted by their GPA from highest to lowest.

In [ ]:
# SQL

In [ ]:
# pandas

**Problem 2:** Determine how many total classes each student has recorded in the `student_grades` table. Include the student's name in the final output. 

In [ ]:
# SQL

In [ ]:
# pandas